In [ ]:
import os
from PIL import Image
import numpy as np
import torch
from tqdm import tqdm
from omegaconf import OmegaConf

from dinov2.inference import generate_embeddings, build_model, view_volume, crop_volume

In [ ]:
def load_volume(folder_path):
    img_paths = [x for x in os.listdir(folder_path) if x.endswith(".png")]
    img_paths.sort(key=lambda x: int(x.split(".")[0]))

    images_stack = []
    for p in img_paths:
        img = Image.open(os.path.join(folder_path, p))
        img = np.array(img)
        images_stack.append(img)

    img = np.stack(images_stack)

    vmin, vmax = -1200.0, 0.0
    hu = (img/255.0) * (vmax - vmin) + vmin
    hu = np.clip(hu, -1000, 1900)
    return torch.from_numpy(hu).float()

In [ ]:
sample_path = "/scratch/VM/radio-foundation/datasets-nodicom/CCCII/Normal/1668/780"
img = load_volume(sample_path)
view_volume(crop_volume(img, k=9), (5.0, 1.0, 1.0))

In [ ]:
config_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/config.yaml"
checkpoint_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/eval/training_79999/teacher_checkpoint.pth"

device = torch.device("cuda")

config = OmegaConf.load(config_path)
model = build_model(checkpoint_path, config, img_size=504, device=device)

In [ ]:
data_path = "/scratch/VM/radio-foundation/datasets-nodicom/CCCII"
output_path = "/scratch/VM/radio-foundation/cache/embeddings/CCCCII"

In [ ]:
data_kwargs = dict(
    fmean = -573.8,
    fstd = 461.3,
    channels = 10,
    img_size = 224,
    patch_size = 14,
    device="cuda",
    block_size=64,
)
class_names = ["CP", "NCP", "Normal"]
for c in class_names:
    print(c)
    base_path = os.path.join(data_path, c)
    os.makedirs(os.path.join(output_path, c), exist_ok=True)
    for id_i in tqdm(os.listdir(base_path)):
        id_i_path = os.path.join(base_path, id_i)
        for id_j in os.listdir(id_i_path):
            scan_path = os.path.join(id_i_path, id_j)
            new_id = f"{id_i:04}_{id_j:04}"

            collated_features = generate_embeddings(
                img,
                model=model,
                **data_kwargs # type: ignore
            )

            torch.save(collated_features, os.path.join(output_path, c, f"{new_id}.pth"))

